# LifeMate · train một seed Qwen3.5-4B SFT trên Colab (L4/A100)
**Máy 1:** `JOB = 'ctl'` (đối chứng 175, seed 2). **Máy 2:** `JOB = 'ren2'` (RenamerV2 bản nộp, seed 0).
Trước khi chạy: tải 2 file zip `lifemate_q35_pkg_asrfold.zip` và `lifemate_q35_pkg_full_ren2.zip` lên Google Drive `MyDrive/lifemate_q35/`.
Bị ngắt: Run all lại, tự chạy tiếp từ checkpoint trên Drive.
Xong (dòng XONG cuối log): tải `MyDrive/lifemate_q35/<NAME>/qwen35-4b-sft-<NAME>-lora.zip` về PC, đặt vào `max-inbox/colab/`.
Ô chạy in một dòng tiến độ mỗi phút (bước/tổng, loss, % GPU, bộ nhớ). 10–20 phút đầu là cài thư viện và tải model nền (9 GB).
Vòng 2: máy phải (sau ctl) → `JOB = 'sv3'`; máy trái (sau ren2) → `JOB = 'sv3full'`. Tải thêm 2 zip `lifemate_q35_pkg_rest_sv3.zip`, `lifemate_q35_pkg_full_sv3.zip` lên Drive.
**Vòng 3 (16/11, F′ = F + vết guard, nhánh đã đạt cổng):** tải `lifemate_q35_pkg_full_guard.zip` lên Drive. Máy xong `sv3` → `JOB = 'fg3'`; máy xong `sv3b` → `JOB = 'fg4'`. Xong thì tải `qwen35-4b-sft-full-guard-s3-lora.zip` / `-s4-lora.zip` về `max-inbox/colab/`.
**Dự phòng (chỉ chạy khi Pro báo trên Relay):** F′ + RenamerV2: `JOB = 'gr0'/'gr1'/'gr2'` (gói `lifemate_q35_pkg_full_guard_ren2.zip`, ~4,1 giờ/seed A100). F′ + synth v3: `'gs0'/'gs1'/'gs2'` (gói `lifemate_q35_pkg_full_guard_sv3.zip`, ~3,2 giờ/seed). Mỗi máy một JOB khác nhau (trùng JOB = ghi đè checkpoint của nhau).
**ƯU TIÊN 1 (16/11 03:45, soup trộn guard+ren2 đạt trên val):** `JOB = 'ren2s1'` và `JOB = 'ren2s2'` (gói `lifemate_q35_pkg_full_ren2.zip`, đã có trên Drive từ vòng 1; ~3,9 giờ/seed A100), mỗi máy một JOB. Xong tải `qwen35-4b-sft-full-ren2-s1-lora.zip` / `-s2-lora.zip` về `max-inbox/colab/`.

In [ ]:
JOB = 'fg3'   # vòng 3 (16/11): máy xong sv3 → 'fg3'; máy xong sv3b → 'fg4'
CFG = {'ctl': ('lifemate_q35_pkg_asrfold.zip', 'train_para_v1_asrfold.jsonl', 'asrfold-ctl-s2', 2),
       'ren2': ('lifemate_q35_pkg_full_ren2.zip', 'train_full_para_v1_asrfold_val_ren2.jsonl', 'full-ren2-s0', 0),
       'ren2s1': ('lifemate_q35_pkg_full_ren2.zip', 'train_full_para_v1_asrfold_val_ren2.jsonl', 'full-ren2-s1', 1),
       'ren2s2': ('lifemate_q35_pkg_full_ren2.zip', 'train_full_para_v1_asrfold_val_ren2.jsonl', 'full-ren2-s2', 2),
       'sv3': ('lifemate_q35_pkg_rest_sv3.zip', 'train_para_v1_asrfold_sv3_400.jsonl', 'rest-sv3-s2', 2),
       'sv3b': ('lifemate_q35_pkg_rest_sv3.zip', 'train_para_v1_asrfold_sv3_400.jsonl', 'rest-sv3-s1', 1),
       'sv3full': ('lifemate_q35_pkg_full_sv3.zip', 'train_full_para_v1_asrfold_val_sv3_400.jsonl', 'full-sv3-s0', 0),
       'fg3': ('lifemate_q35_pkg_full_guard.zip', 'train_full_para_v1_asrfold_val_guard.jsonl', 'full-guard-s3', 3),
       'fg4': ('lifemate_q35_pkg_full_guard.zip', 'train_full_para_v1_asrfold_val_guard.jsonl', 'full-guard-s4', 4),
       'gr0': ('lifemate_q35_pkg_full_guard_ren2.zip', 'train_full_para_v1_asrfold_val_guard_ren2.jsonl', 'full-guard-ren2-s0', 0),
       'gr1': ('lifemate_q35_pkg_full_guard_ren2.zip', 'train_full_para_v1_asrfold_val_guard_ren2.jsonl', 'full-guard-ren2-s1', 1),
       'gr2': ('lifemate_q35_pkg_full_guard_ren2.zip', 'train_full_para_v1_asrfold_val_guard_ren2.jsonl', 'full-guard-ren2-s2', 2),
       'gs0': ('lifemate_q35_pkg_full_guard_sv3.zip', 'train_full_para_v1_asrfold_val_guard_sv3.jsonl', 'full-guard-sv3-s0', 0),
       'gs1': ('lifemate_q35_pkg_full_guard_sv3.zip', 'train_full_para_v1_asrfold_val_guard_sv3.jsonl', 'full-guard-sv3-s1', 1),
       'gs2': ('lifemate_q35_pkg_full_guard_sv3.zip', 'train_full_para_v1_asrfold_val_guard_sv3.jsonl', 'full-guard-sv3-s2', 2)}
PKG, DATA, NAME, SEED = CFG[JOB]
print(PKG, DATA, NAME, SEED)
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil, glob
D = '/content/drive/MyDrive/lifemate_q35'
WORK = f'{D}/{NAME}'
os.makedirs(WORK, exist_ok=True)
shutil.rmtree('/content/pkg', ignore_errors=True)
zipfile.ZipFile(f'{D}/{PKG}').extractall('/content/pkg')
shutil.rmtree('/content/ckin', ignore_errors=True); os.makedirs('/content/ckin')
ck = f'{WORK}/qwen35-4b-sft-{NAME}-ckpt.zip'
if os.path.exists(ck):
    zipfile.ZipFile(ck).extractall('/content/ckin'); print('chạy tiếp từ checkpoint')
else:
    print('chạy từ đầu')
print(sorted(os.listdir('/content/pkg')))

In [ ]:
import subprocess, time, re, os, datetime
os.chdir('/content/pkg')
cmd = f"WORK={WORK} KAGGLE_INPUT=/content/ckin python -u run_sft.py {NAME}={DATA} --deadline 660 --ckpt-every 20 -- --seed {SEED}"
out = open(f'{WORK}/colab_stdout.log', 'a')
p = subprocess.Popen(cmd, shell=True, stdout=out, stderr=subprocess.STDOUT)
t0 = time.time(); last = ''
def tail(path, n=4000):
    try:
        with open(path, 'rb') as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - n)); return f.read().decode('utf-8', 'replace')
    except OSError: return ''
while p.poll() is None:
    time.sleep(60)
    txt = tail(f'{WORK}/{NAME}.log') + tail(f'{WORK}/colab_stdout.log')
    steps = re.findall(r'bước (\d+)/(\d+)[^\n]*?loss ([\d.]+)', txt)
    gpu = subprocess.run("nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
    phase = 'TRAIN' if steps else ('cài thư viện / tải model nền (~10–20 phút đầu)' if time.time() - t0 < 1800 else 'chưa thấy bước train')
    s = f"bước {steps[-1][0]}/{steps[-1][1]} loss {steps[-1][2]}" if steps else ''
    print(f"[{datetime.datetime.now():%H:%M}] {int((time.time()-t0)/60)} phút | {phase} {s} | GPU {gpu}", flush=True)
    if re.search(r'Traceback|LỖI|CUDA out of memory|Error', txt[-1500:]) and txt[-1500:] != last:
        last = txt[-1500:]; print('⚠️ CÓ LỖI, đoạn cuối log:\n' + last, flush=True)
print('KẾT THÚC, mã', p.returncode)
print(tail(f'{WORK}/colab_stdout.log', 1500))
print('XONG' if os.path.exists(f'{WORK}/qwen35-4b-sft-{NAME}-lora.zip') else 'CHƯA XONG: Run all lại để chạy tiếp từ checkpoint')

In [ ]:
print(sorted(glob.glob(f'{WORK}/*')))
!tail -n 5 {WORK}/{NAME}.log